# Step 3b — Fine-tune SPECTER2 with regression head (5 folds x N seeds)

**Goal:** push QWK from 0.628 (TF-IDF anchor) toward 0.70 by **fine-tuning** SPECTER2 directly on the labeled train set as an ordinal regressor on `title + abstract`.

**Why this beats step 3a (frozen SPECTER2 + Ridge):**
- Step 3a froze SPECTER2 and let Ridge map a generic similarity space to labels. Ridge could not recover the ASP-relevance axis cleanly from 768 dims, so OOF QWK collapsed to 0.49 and label-5 predictions collapsed to 14/596.
- Fine-tuning lets the encoder reshape its representation around the *label* signal. With 2494 labeled rows on a BERT-base sized model, this is well within the regime where fine-tuning beats feature-extraction.

**Pipeline:**
1. Upload `asp_data.zip` (same file as step 3a).
2. Build `title [SEP] abstract` inputs.
3. Stratified 5-fold x N-seeds CV. Each fold trains SPECTER2 + a `Linear(768->1)` head with SmoothL1 loss on the float label.
4. Average test predictions across all (folds x seeds) models.
5. Tune 4 thresholds on OOF scores -> labels.
6. Save same artefact set as step 3a (`oof_scores.csv`, `public_scores.csv`, `private_scores.csv`, `metrics.json`, `specter2_finetune_submission.csv`).

**Time budget on A100 (fp16):** ~2 min / fold. 5 folds x 3 seeds = ~30 min. 5 folds x 5 seeds = ~50 min.
**On T4:** roughly 3x slower. Reduce `BATCH=8` and `N_SEEDS=2` if memory is tight.

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q --upgrade "transformers>=4.41" "accelerate>=0.30" scikit-learn pandas numpy scipy

## 2. Upload data

Use the same `asp_data.zip` from step 3a (5 CSVs). Run cell, pick the zip.

In [ ]:
import pathlib, zipfile
from google.colab import files

WORK = pathlib.Path('/content/work')
DATA = WORK / 'data'
OUT = WORK / 'outputs'
RUN_DIR = OUT / 'specter2_finetune'
for d in [WORK, DATA, OUT, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, content in uploaded.items():
    target = WORK / name
    target.write_bytes(content)
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(target) as zf:
            zf.extractall(DATA)
        print('Extracted', name, '->', DATA)

for p in sorted(DATA.glob('*')):
    print(p.name, p.stat().st_size)

## 3. Load + merge

In [ ]:
import pandas as pd, numpy as np, json, re, time, math
from pathlib import Path

DATA = Path('/content/work/data')
RUN_DIR = Path('/content/work/outputs/specter2_finetune')

train = pd.read_csv(DATA / 'train.csv')
public = pd.read_csv(DATA / 'public_test.csv')
private = pd.read_csv(DATA / 'private_test.csv')
sample = pd.read_csv(DATA / 'Test_Submission.csv')
abstracts = pd.read_csv(DATA / 'abstracts_merged_v2.csv')

abs_map = abstracts[['source_split', 'id', 'abstract', 'has_abstract']]

def attach(df, split):
    df = df.copy()
    df['source_split'] = split
    out = df.merge(abs_map, on=['source_split', 'id'], how='left')
    out['abstract'] = out['abstract'].fillna('')
    out['has_abstract'] = out['has_abstract'].fillna(False).astype(bool)
    return out

train_full = attach(train, 'train').reset_index(drop=True)
public_full = attach(public, 'public_test').reset_index(drop=True)
private_full = attach(private, 'private_test').reset_index(drop=True)
print('train', len(train_full), 'public', len(public_full), 'private', len(private_full))
print('train has_abstract:', int(train_full['has_abstract'].sum()))

## 4. Tokenizer + dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'allenai/specter2_base'
MAX_LEN = 256       # almost all our abstracts fit; 256 is ~3x faster than 512.
BATCH_TRAIN = 16    # A100 80GB: can push to 32. T4 16GB: keep 8.
BATCH_EVAL = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
SEP = tokenizer.sep_token

def build_input(title, abstract):
    title = '' if pd.isna(title) else str(title).strip()
    abstract = '' if pd.isna(abstract) else str(abstract).strip()
    return f'{title}{SEP}{abstract}' if abstract else title

class PaperDataset(Dataset):
    def __init__(self, df, with_label):
        self.texts = [build_input(t, a) for t, a in zip(df['title'], df['abstract'])]
        self.labels = df['Label'].astype(np.float32).to_numpy() if with_label else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {'text': self.texts[idx], 'idx': idx}
        if self.labels is not None:
            item['label'] = self.labels[idx]
        return item

def collate(batch):
    texts = [b['text'] for b in batch]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN,
                    return_tensors='pt', return_token_type_ids=False)
    out = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'],
           'idx': torch.tensor([b['idx'] for b in batch], dtype=torch.long)}
    if 'label' in batch[0]:
        out['label'] = torch.tensor([b['label'] for b in batch], dtype=torch.float32)
    return out

print('tokenizer loaded; SEP =', SEP)

## 5. Model: SPECTER2 + regression head

In [ ]:
import torch.nn as nn

class SpecterRegressor(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.encoder.config.hidden_size, 1)
        # Init head with small weights so initial outputs are near 0 -> label 3 mean.
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0]
        return self.head(self.dropout(cls)).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

## 6. Single-fold training + inference

In [ ]:
from sklearn.metrics import cohen_kappa_score
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 5
LR_ENCODER = 2e-5
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

def train_one_fold(train_df, valid_df, public_df, private_df, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader = DataLoader(PaperDataset(train_df, with_label=True),
                              batch_size=BATCH_TRAIN, shuffle=True,
                              collate_fn=collate, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(PaperDataset(valid_df, with_label=True),
                              batch_size=BATCH_EVAL, shuffle=False,
                              collate_fn=collate, num_workers=2, pin_memory=True)
    public_loader = DataLoader(PaperDataset(public_df.assign(Label=0), with_label=False),
                               batch_size=BATCH_EVAL, shuffle=False,
                               collate_fn=collate, num_workers=2, pin_memory=True)
    private_loader = DataLoader(PaperDataset(private_df.assign(Label=0), with_label=False),
                                batch_size=BATCH_EVAL, shuffle=False,
                                collate_fn=collate, num_workers=2, pin_memory=True)

    model = SpecterRegressor().to(device)
    no_decay = ['bias', 'LayerNorm.weight']
    encoder_params = list(model.encoder.named_parameters())
    head_params = list(model.head.named_parameters())
    param_groups = [
        {'params': [p for n, p in encoder_params if not any(nd in n for nd in no_decay)],
         'weight_decay': WEIGHT_DECAY, 'lr': LR_ENCODER},
        {'params': [p for n, p in encoder_params if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0, 'lr': LR_ENCODER},
        {'params': [p for _, p in head_params], 'weight_decay': WEIGHT_DECAY, 'lr': LR_HEAD},
    ]
    optim = AdamW(param_groups)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps)
    loss_fn = nn.SmoothL1Loss(beta=1.0)
    scaler = torch.amp.GradScaler('cuda')

    best_state = None
    best_qwk = -1.0
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        t0 = time.time()
        for step, batch in enumerate(train_loader):
            ids = batch['input_ids'].to(device, non_blocking=True)
            mask = batch['attention_mask'].to(device, non_blocking=True)
            y = batch['label'].to(device, non_blocking=True)
            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                preds = model(ids, mask)
                loss = loss_fn(preds, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optim)
            scaler.update()
            scheduler.step()
            running += loss.item()
        # validation
        val_scores = predict(model, valid_loader)
        rounded = np.clip(np.round(val_scores), 1, 5).astype(int)
        qwk = cohen_kappa_score(valid_df['Label'].astype(int).to_numpy(), rounded, weights='quadratic')
        print(f'  epoch {epoch+1}/{EPOCHS}  loss={running/len(train_loader):.3f}  val_round_QWK={qwk:.4f}  ({time.time()-t0:.1f}s)')
        if qwk > best_qwk:
            best_qwk = qwk
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Load best epoch and predict.
    model.load_state_dict(best_state)
    val_scores = predict(model, valid_loader)
    public_scores = predict(model, public_loader)
    private_scores = predict(model, private_loader)
    return val_scores, public_scores, private_scores, best_qwk

@torch.no_grad()
def predict(model, loader):
    model.eval()
    n = len(loader.dataset)
    out = np.zeros(n, dtype=np.float32)
    for batch in loader:
        ids = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            preds = model(ids, mask).float().cpu().numpy()
        idx = batch['idx'].numpy()
        out[idx] = preds
    return np.clip(out, 1.0, 5.0)

print('train_one_fold ready')

## 7. Run repeated CV (5 folds x N seeds)

Set `SEEDS` below. Default 3 seeds (~30 min on A100). Bump to 5 for the final run.

In [ ]:
from sklearn.model_selection import StratifiedKFold

FOLDS = 5
SEEDS = [252, 253, 254]   # default 3 seeds
# SEEDS = [252, 253, 254, 255, 256]  # full 5 seeds, ~50 min

y_class = train_full['Label'].astype(int).to_numpy()
oof_sum = np.zeros(len(train_full), dtype=np.float64)
oof_count = np.zeros(len(train_full), dtype=np.float64)
public_sum = np.zeros(len(public_full), dtype=np.float64)
private_sum = np.zeros(len(private_full), dtype=np.float64)
n_models = 0
fold_log = []

for seed in SEEDS:
    cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_full, y_class), start=1):
        print(f'\n=== seed={seed} fold={fold}/{FOLDS} ===')
        t0 = time.time()
        train_df = train_full.iloc[tr_idx].reset_index(drop=True)
        valid_df = train_full.iloc[va_idx].reset_index(drop=True)
        val_scores, pub_scores, priv_scores, best_qwk = train_one_fold(
            train_df, valid_df, public_full, private_full, seed * 1000 + fold)
        oof_sum[va_idx] += val_scores
        oof_count[va_idx] += 1.0
        public_sum += pub_scores
        private_sum += priv_scores
        n_models += 1
        fold_log.append({'seed': seed, 'fold': fold, 'best_round_qwk': float(best_qwk),
                         'minutes': round((time.time()-t0)/60, 2)})
        print(f'fold time {(time.time()-t0)/60:.1f} min')

print('\n=== Done. trained', n_models, 'models. ===')
oof_scores = oof_sum / np.clip(oof_count, 1.0, None)
public_scores = public_sum / n_models
private_scores = private_sum / n_models
print('OOF round-QWK:',
      cohen_kappa_score(y_class, np.clip(np.round(oof_scores), 1, 5).astype(int), weights='quadratic'))

## 8. Threshold tuning on OOF

In [ ]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score, mean_absolute_error

def scores_to_labels(scores, thresholds):
    return np.digitize(scores, np.sort(np.asarray(thresholds, dtype=float))) + 1

def tune_thresholds(y_true, oof, seed=42):
    def objective(raw):
        thr = np.sort(raw)
        gap = np.min(np.diff(thr))
        penalty = 0.0 if gap >= 0.03 else (0.03 - gap) * 5.0
        return -cohen_kappa_score(y_true, scores_to_labels(oof, thr), weights='quadratic') + penalty
    bounds = [(1.4, 2.5), (1.8, 2.9), (2.2, 3.4), (2.6, 4.2)]
    res = differential_evolution(objective, bounds, seed=seed, maxiter=120, popsize=15,
                                 polish=True, updating='immediate', workers=1)
    thr = np.sort(res.x)
    return thr, cohen_kappa_score(y_true, scores_to_labels(oof, thr), weights='quadratic')

thresholds, oof_qwk = tune_thresholds(y_class, oof_scores)
oof_pred = scores_to_labels(oof_scores, thresholds)
print('OOF tuned QWK =', round(oof_qwk, 4))
print('thresholds =', thresholds.tolist())
print('OOF MAE =', round(mean_absolute_error(y_class, oof_pred), 4))
print('OOF macro-F1 =', round(f1_score(y_class, oof_pred, average='macro'), 4))
print('OOF distribution =', dict(pd.Series(oof_pred).value_counts().sort_index()))

## 9. Save artefacts + build submission

In [ ]:
public_pred = scores_to_labels(public_scores, thresholds)
private_pred = scores_to_labels(private_scores, thresholds)

metrics = {
    'method': 'specter2_finetune',
    'model': MODEL_NAME,
    'folds': FOLDS,
    'seeds': SEEDS,
    'epochs': EPOCHS,
    'max_len': MAX_LEN,
    'batch_train': BATCH_TRAIN,
    'lr_encoder': LR_ENCODER,
    'lr_head': LR_HEAD,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'oof_qwk': float(oof_qwk),
    'oof_mae': float(mean_absolute_error(y_class, oof_pred)),
    'oof_macro_f1': float(f1_score(y_class, oof_pred, average='macro')),
    'thresholds': [float(v) for v in thresholds],
    'label_distribution_combined': {int(k): int(v) for k, v in pd.Series(
        np.concatenate([public_pred, private_pred])).value_counts().sort_index().items()},
    'label_distribution_public': {int(k): int(v) for k, v in pd.Series(public_pred).value_counts().sort_index().items()},
    'label_distribution_private': {int(k): int(v) for k, v in pd.Series(private_pred).value_counts().sort_index().items()},
    'fold_log': fold_log,
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

pd.DataFrame({'id': train_full['id'], 'Label': y_class,
              'oof_score': oof_scores, 'oof_pred': oof_pred}).to_csv(
    RUN_DIR / 'oof_scores.csv', index=False)
pd.DataFrame({'id': public_full['id'], 'score': public_scores,
              'pred': public_pred}).to_csv(RUN_DIR / 'public_scores.csv', index=False)
pd.DataFrame({'id': private_full['id'], 'score': private_scores,
              'pred': private_pred}).to_csv(RUN_DIR / 'private_scores.csv', index=False)

combo = pd.concat([
    pd.DataFrame({'id': public_full['id'], 'Label': public_pred}),
    pd.DataFrame({'id': private_full['id'], 'Label': private_pred}),
], ignore_index=True)
submission = sample[['id']].merge(combo, on='id', how='left')
assert submission['Label'].notna().all() and submission.shape[0] == len(sample)
submission['Label'] = submission['Label'].astype(int)
submission.to_csv(RUN_DIR / 'specter2_finetune_submission.csv', index=False)
print('submission rows =', len(submission))
print(submission.head())

## 10. Zip outputs + download

Download `specter2_finetune_outputs.zip`, extract into `outputs/specter2_finetune/` in the local repo before running step 5 (stacking).

In [ ]:
zip_path = pathlib.Path('/content/specter2_finetune_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.iterdir():
        zf.write(p, arcname=f'specter2_finetune/{p.name}')
print('zipped:', zip_path, 'size MB =', round(zip_path.stat().st_size / 1e6, 2))
files.download(str(zip_path))